In [ ]:
import os
os.environ["HUGGINGFACEHUB_API_TOKEN"] = "insert_key"

from huggingface_hub import login
login(token="insert_key")

In [ ]:
"""
This is a simple application for sentence embeddings: semantic search

We have a corpus with various sentences. Then, for a given query sentence,
we want to find the most similar sentence in this corpus.

This script outputs for various queries the top 5 most similar sentences in the corpus.
"""

import torch
import time
import nltk

from nltk import sent_tokenize
from sentence_transformers import CrossEncoder
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")
cross_encoder = CrossEncoder("cross-encoder/mmarco-mMiniLMv2-L12-H384-v1")

# Corpus with example documents
corpus = [
    "A empresa deve pagar as diferenças salariais de setembro a novembro de 2025 em parcela única, integrando a base de cálculo das verbas rescisórias, e comunicar o empregado no prazo máximo de 10 dias da assinatura da convenção para recebimento.",
    "Para salários acima do piso e abaixo do teto, abono de 20% sobre o salário de agosto/2025, pago em duas parcelas (janeiro e fevereiro/2026). Para salários acima do teto, abono fixo de R$ 2.200,00, também em duas parcelas, como alternativa ao parcelamento das diferenças.",
    "As diferenças salariais de setembro a novembro/2025 devem ser parceladas em até três vezes (janeiro, fevereiro e março/2026). Para admitidos entre 01/09/2024 e 30/08/2025, deve-se observar a tabela proporcional de reajuste da cláusula 2.",
    "Se as férias foram concedidas entre 1º/09/2025 e a assinatura da convenção, as diferenças salariais decorrentes do reajuste devem ser pagas na folha de janeiro/2026.",
    "O 13º salário deve ser calculado com base no salário reajustado integralmente, conforme previsto no caput da cláusula de reajustamento, sem parcelamento ou compensações.",
    "Empresas com até 20 empregados podem aderir ao REPIS, com pisos reduzidos (ex.: R$ 1.580,00 para auxiliares), mediante Certidão de Adesão e cumprimento da cláusula 60. Caso contrário, devem seguir os pisos das cláusulas 4 e 5 (empresas com mais de 20).",
    "Comissionistas puros têm garantia mínima de R$ 2.536,00 (empresas >20 empregados). O repouso semanal é calculado sobre a média das comissões, e as horas extras são apuradas com base na média dos últimos três meses, com adicional de 60%.",
    "Empresas enquadradas como MEI, ME ou EPP (até 20 empregados) que aderirem ao REPIS têm pisos diferenciados: R$ 1.580,00 para office-boy, R$ 1.977,00 para demais e garantia de comissionista de R$ 2.372,00, além de obrigatoriedade do Plano de Assistência (cláusula 60).",
    "Será descontado 1% do salário, limitado a R$ 50,00 mensais, a partir de setembro/2025, em 12 parcelas (dez/2025 a nov/2026). O empregado pode exercer oposição em até 10 dias úteis após a assinatura da convenção.",
    "Mediante acordo individual, as horas extras podem ser compensadas em até 12 meses, limitadas a 2 horas diárias e saldo máximo de 150 horas. Não compensadas no prazo, são pagas com adicional de 60%. O empregado deve receber comprovante mensal do saldo.",
]
# Use "convert_to_tensor=True" to keep the tensors on GPU (if available)
corpus_embeddings = embedder.encode_document(corpus, convert_to_tensor=True)

# Query sentences:
queries = [
    "Em casos de rescisão de contrato, como a empresa deve proceder, no tocante a diferenças salariais?",
    "Quais as particularidades para cada caso do abono pecuniário?",
    "Quais obrigações a empresa deve cumprir, em caso de parcelamento da rescisão?",
    "Quais as diferenças de aplicação do reajuste, conforme o período de férias concedidas ao colaborador?",
    "Como funciona o pagamento do 13º salário após o reajuste?",
    "Quais regras são aplicáveis para empresas com até 20 empregados?",
    "Quais são as particularidades de remuneração para comissionistas?",
    "Quais as particularidades de remuneração para MEIs, MEs e EPPs?",
    "Com o reajuste, como funcionará a contribuição assistencial dos empregados?",
    "Quais são as diretrizes para compensação de horas de trabalho (banco de horas)?",
]

# Find the closest 5 sentences of the corpus for each query sentence based on cosine similarity
top_k = min(5, len(corpus))
for query in queries:
    query_embedding = embedder.encode_query(query, convert_to_tensor=True)

    # We use cosine-similarity and torch.topk to find the highest 5 scores
    similarity_scores = embedder.similarity(query_embedding, corpus_embeddings)[0]
    scores, indices = torch.topk(similarity_scores, k=top_k)

    print("\nQuery:", query)
    print("Top 5 most similar sentences in corpus:")

    for score, idx in zip(scores, indices):
        print(f"(Score: {score:.4f})", corpus[idx])
    
    # 2. Cross-encoder re-ranking
    # Prepare (query, document) pairs for the top candidates
    top_docs = [corpus[idx] for idx in indices]
    pairs = [(query, doc) for doc in top_docs]
    
    # Predict cross-encoder scores (higher = more relevant)
    cross_scores = cross_encoder.predict(pairs)
    
    # Sort documents by cross-encoder score descending
    ranked_results = sorted(zip(cross_scores, top_docs), key=lambda x: x[0], reverse=True)
    
    print("\nRe-ranked by cross-encoder:")
    for score, doc in ranked_results:
        print(f"  Cross-Encoder Score: {score:.4f}  |  {doc}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: cross-encoder/mmarco-mMiniLMv2-L12-H384-v1
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Query: Em casos de rescisão de contrato, como a empresa deve proceder, no tocante a diferenças salariais?
Top 5 most similar sentences in corpus:
(Score: 0.5970) A empresa deve pagar as diferenças salariais de setembro a novembro de 2025 em parcela única, integrando a base de cálculo das verbas rescisórias, e comunicar o empregado no prazo máximo de 10 dias da assinatura da convenção para recebimento.
(Score: 0.5432) Empresas com até 20 empregados podem aderir ao REPIS, com pisos reduzidos (ex.: R$ 1.580,00 para auxiliares), mediante Certidão de Adesão e cumprimento da cláusula 60. Caso contrário, devem seguir os pisos das cláusulas 4 e 5 (empresas com mais de 20).
(Score: 0.5171) Comissionistas puros têm garantia mínima de R$ 2.536,00 (empresas >20 empregados). O repouso semanal é calculado sobre a média das comissões, e as horas extras são apuradas com base na média dos últimos três meses, com adicional de 60%.
(Score: 0.4727) Mediante acordo individual, as horas extras podem ser co